# 4044. Count Good Cyclic Rotations — Nhập môn Sliding Window vòng tròn + Prefix Sum

Bài toán LeetCode (Medium):

```python
class Solution:
    def countGoodRotations(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu của notebook

Sau notebook này, bạn sẽ hiểu:

- **Xoay vòng** (cyclic rotation) một mảng là gì, cách tạo bằng slicing.
- Đổi điều kiện "nửa đầu > nửa sau" thành `2 * first > total` — chỉ cần theo dõi **một** tổng.
- Lời giải Brute Force `O(n²)`.
- **Sliding window** (cửa sổ trượt) trên mảng **vòng tròn** dùng phép `%` → `O(n)`.
- **Prefix sum trên mảng nhân đôi** → `O(n)`.
- Mẹo sáng tạo: **hai phép xoay cách nhau `n/2` là "đối xứng"** → chỉ cần xét một nửa.
- Trace, code chuẩn LeetCode, lỗi hay mắc, biến thể, bài tập có đáp án.

# 1. Đề bài nói gì?

Cho mảng số nguyên `nums` có độ dài **chẵn** `n`.

Một **phép xoay vòng**: chọn một tiền tố (prefix) độ dài từ `0` đến `n - 1`, **chuyển nó xuống cuối** mảng, giữ nguyên thứ tự.

Phép xoay là **tốt** nếu tổng `n/2` phần tử **đầu** **lớn hơn hẳn** tổng `n/2` phần tử **cuối**.

Đếm số phép xoay tốt.

### Ví dụ 1: `nums = [1, 2, 3, 4, 5, 6]` → `3`

| k (độ dài tiền tố chuyển đi) | Mảng sau xoay | Tổng nửa đầu | Tổng nửa sau | Tốt? |
|---|---|---:|---:|---|
| 0 | `[1, 2, 3, 4, 5, 6]` | 6 | 15 | ✘ |
| 1 | `[2, 3, 4, 5, 6, 1]` | 9 | 12 | ✘ |
| 2 | `[3, 4, 5, 6, 1, 2]` | 12 | 9 | ✔ |
| 3 | `[4, 5, 6, 1, 2, 3]` | 15 | 6 | ✔ |
| 4 | `[5, 6, 1, 2, 3, 4]` | 12 | 9 | ✔ |
| 5 | `[6, 1, 2, 3, 4, 5]` | 9 | 12 | ✘ |

### Ví dụ 2: `nums = [1, 2, 1, 2]` → `0` (mọi phép xoay đều cho hai nửa bằng nhau)

### Ràng buộc

- `2 <= n <= 10^5`, `n` chẵn
- `1 <= nums[i] <= 10^9`

> Lưu ý: ta đếm **phép xoay** (theo `k`), không đếm **mảng kết quả khác nhau**. Ở ví dụ 2, phép xoay `k = 0` và `k = 2` cho cùng mảng nhưng vẫn là hai phép xoay khác nhau.

# 2. Nền tảng Python — Xoay vòng bằng slicing

Chuyển tiền tố độ dài `k` xuống cuối:

```python
nums[k:] + nums[:k]
```

- `nums[k:]`: từ vị trí `k` đến hết.
- `nums[:k]`: `k` phần tử đầu.

In [1]:
nums = [1, 2, 3, 4, 5, 6]
for k in range(len(nums)):
    print(f"k = {k}: {nums[k:]} + {nums[:k]} = {nums[k:] + nums[:k]}")

k = 0: [1, 2, 3, 4, 5, 6] + [] = [1, 2, 3, 4, 5, 6]
k = 1: [2, 3, 4, 5, 6] + [1] = [2, 3, 4, 5, 6, 1]
k = 2: [3, 4, 5, 6] + [1, 2] = [3, 4, 5, 6, 1, 2]
k = 3: [4, 5, 6] + [1, 2, 3] = [4, 5, 6, 1, 2, 3]
k = 4: [5, 6] + [1, 2, 3, 4] = [5, 6, 1, 2, 3, 4]
k = 5: [6] + [1, 2, 3, 4, 5] = [6, 1, 2, 3, 4, 5]


# 3. Biến hai tổng thành một

Gọi `total = sum(nums)` — tổng **không đổi** dù xoay thế nào. Gọi `first` = tổng nửa đầu. Khi đó tổng nửa sau là `total - first`.

Điều kiện tốt:

```text
first > total - first   ⇔   2 * first > total
```

Ta chỉ cần theo dõi **một** con số `first` cho mỗi phép xoay.

> Vì sao viết `2 * first > total` thay vì `first > total / 2`? Phép `/` cho số thực (`float`). Với số lớn, `float` có thể làm tròn sai. Nhân 2 giữ mọi thứ là **số nguyên chính xác**.

In [2]:
nums = [1, 2, 3, 4, 5, 6]
total = sum(nums)
rotated = nums[2:] + nums[:2]
first = sum(rotated[:3])
print("total =", total, "| first =", first, "| 2*first > total ?", 2 * first > total)

total = 21 | first = 12 | 2*first > total ? True


# 4. Cách 1 — Brute Force

Với mỗi `k`: tạo mảng xoay, tính tổng nửa đầu, so sánh.

In [3]:
def count_good_bruteforce(nums):
    n = len(nums)
    half = n // 2
    total = sum(nums)
    count = 0
    for k in range(n):
        rotated = nums[k:] + nums[:k]
        first = sum(rotated[:half])
        if 2 * first > total:
            count += 1
    return count


print(count_good_bruteforce([1, 2, 3, 4, 5, 6]))   # 3
print(count_good_bruteforce([1, 2, 1, 2]))         # 0

3
0


Mỗi phép xoay tốn `O(n)` (tạo mảng + tính tổng) → tổng `O(n²)`. Với `n = 10^5`: `10^10` bước → quá chậm.

**Nút thắt:** hai phép xoay liên tiếp có nửa đầu **gần như giống nhau**. Tính lại từ đầu là lãng phí.

# 5. Quan sát — Nửa đầu là một "cửa sổ" trên vòng tròn

Hãy hình dung `nums` xếp thành **vòng tròn**. Nửa đầu của phép xoay `k` chính là `n/2` phần tử **liên tiếp trên vòng tròn**, bắt đầu từ vị trí `k`:

```text
vị trí trên vòng: nums[k], nums[k+1], ..., nums[k + n/2 - 1]   (chỉ số lấy % n)
```

Khi `k` tăng thêm 1, "cửa sổ" **trượt** một bước:

- **bỏ** phần tử `nums[k]` ở đầu cửa sổ,
- **thêm** phần tử `nums[(k + n/2) % n]` ở cuối cửa sổ.

```text
k = 0: [1  2  3] 4  5  6        first = 6
k = 1:  1 [2  3  4] 5  6        first = 6 - 1 + 4 = 9
k = 2:  1  2 [3  4  5] 6        first = 9 - 2 + 5 = 12
k = 3:  1  2  3 [4  5  6]       first = 12 - 3 + 6 = 15
k = 4:  1] 2  3  4 [5  6        first = 15 - 4 + 1 = 12   (cửa sổ vòng qua đầu mảng)
k = 5:  1  2] 3  4  5 [6        first = 12 - 5 + 2 = 9
```

Đây là kỹ thuật **Sliding Window** — mỗi bước chỉ tốn `O(1)`.

# 6. Nền tảng Python — Chỉ số vòng tròn với `%`

`i % n` luôn nằm trong `0..n-1`. Khi `i` vượt quá cuối mảng, `%` đưa nó **quay về đầu**:

In [4]:
n = 6
for i in range(4, 10):
    print(f"{i} % {n} = {i % n}")

4 % 6 = 4
5 % 6 = 5
6 % 6 = 0
7 % 6 = 1
8 % 6 = 2
9 % 6 = 3


# 7. Cách 2 — Sliding Window vòng tròn: `O(n)`

```text
first = sum(nums[0 : n/2])           # cửa sổ của k = 0
với k = 0, 1, ..., n-1:
    nếu 2 * first > total: count += 1
    first += nums[(k + n/2) % n] - nums[k]      # trượt sang k + 1
```

In [5]:
def count_good_sliding(nums):
    n = len(nums)
    half = n // 2
    total = sum(nums)
    first = sum(nums[:half])
    count = 0
    for k in range(n):
        if 2 * first > total:
            count += 1
        first += nums[(k + half) % n] - nums[k]     # chuẩn bị cho phép xoay k + 1
    return count


print(count_good_sliding([1, 2, 3, 4, 5, 6]))   # 3
print(count_good_sliding([1, 2, 1, 2]))         # 0

3
0


## Mô phỏng từng bước

`nums = [1, 2, 3, 4, 5, 6]`, `half = 3`, `total = 21`.

| k | first (cửa sổ k) | 2·first > 21? | count | bỏ `nums[k]` | thêm `nums[(k+3)%6]` | first cho k+1 |
|---|---:|---|---:|---:|---:|---:|
| 0 | 6 | 12 ✘ | 0 | 1 | 4 | 9 |
| 1 | 9 | 18 ✘ | 0 | 2 | 5 | 12 |
| 2 | 12 | 24 ✔ | 1 | 3 | 6 | 15 |
| 3 | 15 | 30 ✔ | 2 | 4 | 1 | 12 |
| 4 | 12 | 24 ✔ | 3 | 5 | 2 | 9 |
| 5 | 9 | 18 ✘ | 3 | 6 | 3 | 6 |

Đáp án `3` ✔. Chú ý dòng cuối: cửa sổ quay về đúng giá trị ban đầu `6` — dấu hiệu ta trượt đúng.

In [6]:
def trace_sliding(nums):
    n, half, total = len(nums), len(nums) // 2, sum(nums)
    first = sum(nums[:half])
    count = 0
    print(f"half = {half}, total = {total}")
    for k in range(n):
        good = 2 * first > total
        count += good
        out_val, in_val = nums[k], nums[(k + half) % n]
        print(f"k={k}: first={first:>3}, 2*first={2 * first:>3} {'>' if good else '<='} {total} "
              f"-> {'TỐT' if good else '   '} | count={count} | bỏ {out_val}, thêm {in_val}")
        first += in_val - out_val
    return count

trace_sliding([1, 2, 3, 4, 5, 6])

half = 3, total = 21
k=0: first=  6, 2*first= 12 <= 21 ->     | count=0 | bỏ 1, thêm 4
k=1: first=  9, 2*first= 18 <= 21 ->     | count=0 | bỏ 2, thêm 5
k=2: first= 12, 2*first= 24 > 21 -> TỐT | count=1 | bỏ 3, thêm 6
k=3: first= 15, 2*first= 30 > 21 -> TỐT | count=2 | bỏ 4, thêm 1
k=4: first= 12, 2*first= 24 > 21 -> TỐT | count=3 | bỏ 5, thêm 2
k=5: first=  9, 2*first= 18 <= 21 ->     | count=3 | bỏ 6, thêm 3


3

# 8. Cách 3 — Prefix Sum trên mảng nhân đôi

Một mẹo cực phổ biến với bài **vòng tròn**: **nối mảng với chính nó** `doubled = nums + nums`. Khi đó mọi đoạn liên tiếp trên vòng tròn (độ dài `<= n`) đều là một đoạn **bình thường** (không vòng) trong `doubled`.

Nửa đầu của phép xoay `k` = `doubled[k : k + half]`. Dùng prefix sum `P` của `doubled`:

```text
first(k) = P[k + half] - P[k]
```

(Nhắc lại: `P[t]` = tổng `t` phần tử đầu; tổng đoạn `[a, b)` = `P[b] - P[a]`.)

In [7]:
from itertools import accumulate

nums = [1, 2, 3, 4, 5, 6]
doubled = nums + nums
P = [0] + list(accumulate(doubled))
print("doubled =", doubled)
print("P       =", P)
print("first(4) = P[7] - P[4] =", P[7] - P[4], "| kiểm tra:", sum(doubled[4:7]))

doubled = [1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6]
P       = [0, 1, 3, 6, 10, 15, 21, 22, 24, 27, 31, 36, 42]
first(4) = P[7] - P[4] = 12 | kiểm tra: 12


`itertools.accumulate(arr)` trả về các **tổng cộng dồn**: `[a0, a0+a1, a0+a1+a2, ...]`. Thêm `[0] +` ở đầu để có quy ước `P[0] = 0`.

In [8]:
def count_good_prefix(nums):
    n = len(nums)
    half = n // 2
    total = sum(nums)
    P = [0] + list(accumulate(nums + nums))
    return sum(1 for k in range(n) if 2 * (P[k + half] - P[k]) > total)


print(count_good_prefix([1, 2, 3, 4, 5, 6]))   # 3
print(count_good_prefix([1, 2, 1, 2]))         # 0

3
0


So với sliding window, cách này dùng thêm `O(n)` bộ nhớ cho `P`, nhưng **mỗi `first(k)` tính độc lập** — tiện khi cần truy vấn tuỳ ý (không theo thứ tự).

# 9. Cách 4 — Mẹo sáng tạo: các phép xoay đi theo cặp

Nhìn lại bảng ví dụ 1:

| k | nửa đầu | nửa sau |
|---|---:|---:|
| 0 | 6 | 15 |
| 3 | **15** | **6** |

Phép xoay `k` và `k + n/2` **hoán đổi hai nửa cho nhau**! Lý do: xoay thêm `n/2` bước nghĩa là chuyển đúng nửa đầu xuống cuối.

Vậy với mỗi cặp `(k, k + n/2)` (với `k = 0 .. n/2 - 1`):

- nếu hai nửa **khác** nhau → **đúng một** trong hai phép xoay là tốt;
- nếu hai nửa **bằng** nhau → không phép nào tốt.

> **Đáp án = số `k` trong `0 .. n/2 - 1` mà `2 * first(k) != total`.**

Chỉ cần xét **một nửa** số phép xoay!

In [9]:
def count_good_pairs(nums):
    n = len(nums)
    half = n // 2
    total = sum(nums)
    first = sum(nums[:half])
    count = 0
    for k in range(half):                       # chỉ nửa đầu số phép xoay
        if 2 * first != total:
            count += 1                          # đúng 1 trong 2 phép xoay (k, k + half) là tốt
        first += nums[k + half] - nums[k]       # k + half < n nên không cần %
    return count


print(count_good_pairs([1, 2, 3, 4, 5, 6]))   # 3
print(count_good_pairs([1, 2, 1, 2]))         # 0

3
0


Một điểm thưởng: vì `k < half`, chỉ số `k + half` luôn `< n` → **không cần `%`**.

Cách này đẹp vì nó cho ta một **kết quả lý thuyết**: đáp án luôn `<= n/2`, và bằng `n/2` khi không có phép xoay nào cho hai nửa bằng nhau (ví dụ khi `total` lẻ).

In [10]:
# Kiểm chứng tính chất "cặp đối xứng"
nums = [7, 1, 8, 2, 8, 1, 8, 2]
n, half = len(nums), len(nums) // 2
for k in range(half):
    r1 = nums[k:] + nums[:k]
    r2 = nums[k + half:] + nums[:k + half]
    print(f"k={k}: xoay {k} có hai nửa ({sum(r1[:half])}, {sum(r1[half:])}), "
          f"xoay {k + half} có ({sum(r2[:half])}, {sum(r2[half:])})")

k=0: xoay 0 có hai nửa (18, 19), xoay 4 có (19, 18)
k=1: xoay 1 có hai nửa (19, 18), xoay 5 có (18, 19)
k=2: xoay 2 có hai nửa (19, 18), xoay 6 có (18, 19)
k=3: xoay 3 có hai nửa (19, 18), xoay 7 có (18, 19)


# 10. Lời giải LeetCode chuẩn

Chọn **cách 2** (sliding window) làm bản nộp vì trực tiếp và dễ kiểm tra; cách 4 cũng hoàn toàn hợp lệ.

In [11]:
class Solution:
    def countGoodRotations(self, nums: list[int]) -> int:
        n = len(nums)
        half = n // 2
        total = sum(nums)
        first = sum(nums[:half])           # nửa đầu của phép xoay k = 0
        count = 0
        for k in range(n):
            if 2 * first > total:
                count += 1
            first += nums[(k + half) % n] - nums[k]
        return count

In [12]:
sol = Solution()

tests = [
    ([1, 2, 3, 4, 5, 6], 3),
    ([1, 2, 1, 2], 0),
    ([5, 1], 1),                       # k=0: 5 > 1 ✔ ; k=1: 1 > 5 ✘
    ([3, 3], 0),
    ([1, 1, 1, 2], 2),                 # total = 5 lẻ -> luôn n/2 = 2
    ([10**9, 1, 10**9, 1], 0),
    ([10**9, 10**9, 1, 1], 1),         # chỉ k = 0 tốt; k = 1, 3 cho hai nửa bằng nhau
]
for nums, expected in tests:
    result = sol.countGoodRotations(nums)
    print(f"{str(nums):34} -> {result} | expected {expected} | {'PASS' if result == expected else 'FAIL'}")
    assert result == expected

[1, 2, 3, 4, 5, 6]                 -> 3 | expected 3 | PASS
[1, 2, 1, 2]                       -> 0 | expected 0 | PASS
[5, 1]                             -> 1 | expected 1 | PASS
[3, 3]                             -> 0 | expected 0 | PASS
[1, 1, 1, 2]                       -> 2 | expected 2 | PASS
[1000000000, 1, 1000000000, 1]     -> 0 | expected 0 | PASS
[1000000000, 1000000000, 1, 1]     -> 1 | expected 1 | PASS


In [13]:
import random, time

random.seed(44)
for _ in range(3000):
    nums = [random.randint(1, 5) for _ in range(2 * random.randint(1, 7))]
    b = count_good_bruteforce(nums)
    assert count_good_sliding(nums) == b
    assert count_good_prefix(nums) == b
    assert count_good_pairs(nums) == b
    assert sol.countGoodRotations(nums) == b
print("4 cách khớp nhau trên 3000 mảng ngẫu nhiên ✔")

big = [random.randint(1, 10**9) for _ in range(10**5)]
for f in (count_good_sliding, count_good_prefix, count_good_pairs):
    t0 = time.time()
    f(big)
    print(f"{f.__name__:20}: {time.time() - t0:.3f} giây")

4 cách khớp nhau trên 3000 mảng ngẫu nhiên ✔
count_good_sliding  : 0.012 giây
count_good_prefix   : 0.020 giây
count_good_pairs    : 0.006 giây


# 11. Vì sao thuật toán đúng?

**Cách 2:** Bất biến của vòng lặp: ở đầu vòng `k`, `first = Σ nums[(k + t) % n]` với `t = 0 .. half-1` — đúng là tổng nửa đầu của phép xoay `k`. Ban đầu (`k = 0`) đúng do `first = sum(nums[:half])`. Bước trượt bỏ `nums[k]` (phần tử `t = 0`) và thêm `nums[(k + half) % n]` (phần tử `t = half` của cửa sổ mới) → đúng cho `k + 1`. Điều kiện tốt `first > total - first` ⇔ `2·first > total`.

**Cách 4:** Phép xoay `k + half` = lấy mảng của phép xoay `k` rồi chuyển nửa đầu xuống cuối → nửa đầu và nửa sau hoán đổi. Gọi `a = first(k)`, `b = total - a`. Phép xoay `k` tốt ⇔ `a > b`; phép xoay `k + half` tốt ⇔ `b > a`. Đúng một điều xảy ra khi `a ≠ b`, không điều nào khi `a = b`. Các cặp `(k, k + half)` với `k = 0..half-1` phủ **đúng một lần** mọi `k` trong `0..n-1`.

# 12. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| 1. Brute Force | `O(n²)` | `O(n)` |
| 2. Sliding window vòng tròn | `O(n)` | `O(1)` |
| 3. Prefix sum mảng nhân đôi | `O(n)` | `O(n)` |
| 4. Cặp đối xứng | `O(n)` (vòng lặp chỉ `n/2` bước) | `O(1)` |

# 13. Những lỗi thường gặp

## Lỗi 1 — Dùng `>=` thay vì `>`

Đề yêu cầu **lớn hơn hẳn**. Với `[1, 2, 1, 2]` dùng `>=` sẽ ra `4` thay vì `0`.

## Lỗi 2 — Quên `%` khi trượt cửa sổ

`nums[k + half]` sẽ `IndexError` khi `k + half >= n`.

## Lỗi 3 — So sánh bằng số thực `first > total / 2`

Với số rất lớn có thể sai do làm tròn. Python dùng float 64-bit; số nguyên tới `10^14` vẫn biểu diễn chính xác nên ở bài này thường không sai, nhưng đây là **thói quen xấu** — trong nhiều bài khác nó gây lỗi thật.

## Lỗi 4 — Trượt cửa sổ **trước** khi kiểm tra

Nếu cập nhật `first` **trước** rồi mới kiểm tra, vòng lặp thực chất kiểm tra các phép xoay `1, 2, ..., n` thay vì `0, 1, ..., n-1`. Vì phép xoay `n` trùng với phép xoay `0`, kết quả ở bài này **tình cờ** vẫn đúng — nhưng logic đã sai, và sẽ hỏng ngay khi đề đổi một chút (ví dụ chỉ xét `k` từ `0` đến `n/2 - 1` như cách 4). Hãy giữ thứ tự: **kiểm tra cửa sổ hiện tại → rồi mới trượt**.

In [14]:
def wrong_non_strict(nums):
    n, half, total = len(nums), len(nums) // 2, sum(nums)
    first = sum(nums[:half])
    count = 0
    for k in range(n):
        if 2 * first >= total:                  # SAI: >=
            count += 1
        first += nums[(k + half) % n] - nums[k]
    return count

print("Lỗi 1 | sai:", wrong_non_strict([1, 2, 1, 2]), "| đúng:", sol.countGoodRotations([1, 2, 1, 2]))
print("Float có thể làm tròn:", (10**17 + 1) / 2 == 10**17 / 2, "<- hai số khác nhau nhưng float coi là bằng")

Lỗi 1 | sai: 4 | đúng: 0
Float có thể làm tròn: True <- hai số khác nhau nhưng float coi là bằng


# 14. Mẹo nhận diện pattern

- **"Xoay vòng"**, **"mảng vòng tròn"**, **"circular"** → nghĩ tới:
  - chỉ số `% n`, hoặc
  - **nhân đôi mảng** `nums + nums`.
- **Tổng các đoạn liên tiếp cùng độ dài** → **sliding window** (thêm một, bớt một).
- **So sánh hai phần bù nhau** (nửa đầu vs nửa sau, chọn vs không chọn) → dùng `total`: `A > total - A` ⇔ `2A > total`.
- Tìm **tính đối xứng** giữa các trường hợp (ở đây: xoay `k` ↔ xoay `k + n/2`) để giảm một nửa công việc hoặc ra công thức.

# 15. Biến thể — Cửa sổ độ dài `m` bất kỳ

Đếm phép xoay mà tổng `m` phần tử đầu **lớn hơn** tổng `n - m` phần tử còn lại. Sliding window vẫn dùng y hệt, chỉ đổi `half` thành `m`. (Mẹo cặp đối xứng **không** còn áp dụng khi `m ≠ n/2`.)

In [15]:
def count_good_window_m(nums, m):
    n, total = len(nums), sum(nums)
    first = sum(nums[:m])
    count = 0
    for k in range(n):
        if 2 * first > total:
            count += 1
        first += nums[(k + m) % n] - nums[k]
    return count

def count_good_window_m_bruteforce(nums, m):
    n, total = len(nums), sum(nums)
    return sum(1 for k in range(n) if 2 * sum((nums[k:] + nums[:k])[:m]) > total)

random.seed(15)
for _ in range(1000):
    nums = [random.randint(1, 9) for _ in range(random.randint(1, 8))]
    m = random.randint(1, len(nums))
    assert count_good_window_m(nums, m) == count_good_window_m_bruteforce(nums, m)
print("Biến thể cửa sổ độ dài m: khớp brute force ✔", count_good_window_m([1, 2, 3, 4, 5, 6], 2))

Biến thể cửa sổ độ dài m: khớp brute force ✔ 1


# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `nums = [3, 1, 2, 5]`:

1. Liệt kê 4 phép xoay, tổng hai nửa, tốt hay không.
2. Lập bảng trace sliding window như mục 7.
3. Kiểm tra mẹo cặp đối xứng: chỉ cần xét `k = 0, 1`.

## Bài 2 — Dự đoán output

```python
[1, 2, 3, 4][3:] + [1, 2, 3, 4][:3]
list(accumulate([2, 2, 5]))
[(k + 3) % 6 for k in range(6)]
```

In [16]:
print([1, 2, 3, 4][3:] + [1, 2, 3, 4][:3])
print(list(accumulate([2, 2, 5])))
print([(k + 3) % 6 for k in range(6)])

[4, 1, 2, 3]
[2, 4, 9]
[3, 4, 5, 0, 1, 2]


## Bài 3 — Điền chỗ trống: sliding window

In [17]:
def exercise_3_sliding(nums):
    n = len(nums)
    half = n // 2
    total = sum(nums)
    first = sum(nums[:half])
    count = 0
    # TODO: với k = 0..n-1:
    #   - nếu 2 * first > total: count += 1
    #   - first += nums[(k + half) % n] - nums[k]
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")

# assert exercise_3_sliding([1, 2, 3, 4, 5, 6]) == 3

## Bài 4 — Tự viết cách 4 (cặp đối xứng)

In [18]:
def exercise_4_pairs(nums):
    # TODO
    raise NotImplementedError

## Bài 5 — Biến thể: "không tệ hơn"

Đếm phép xoay mà nửa đầu **lớn hơn hoặc bằng** nửa sau. Hãy dùng mẹo cặp đối xứng để suy ra công thức theo số cặp bằng nhau.

In [19]:
def exercise_5_at_least(nums):
    # TODO
    raise NotImplementedError

## Bài 6 — Mini challenge: phép xoay "tốt nhất"

Trả về `k` (nhỏ nhất nếu có nhiều) sao cho hiệu `nửa đầu - nửa sau` **lớn nhất**.

Ví dụ: `[1, 2, 3, 4, 5, 6]` → `3` (hiệu `15 - 6 = 9`).

In [20]:
def exercise_6_best_rotation(nums):
    # TODO
    raise NotImplementedError

## Gợi ý theo mức độ

**Bài 4**
- Mức 1: chỉ duyệt `k` từ `0` tới `half - 1`.
- Mức 2: đếm khi `2 * first != total`.
- Mức 3: trượt bằng `first += nums[k + half] - nums[k]` (không cần `%`).

**Bài 5**
- Mức 1: một cặp `(k, k + half)` có hai nửa khác nhau → đóng góp 1; bằng nhau → đóng góp **2** (cả hai đều "≥").
- Mức 2: đáp án `= half + (số cặp bằng nhau)`.

**Bài 6**
- Mức 1: hiệu `= first - (total - first) = 2·first - total` → tối đa hoá `first`.
- Mức 2: sliding window, nhớ `k` có `first` lớn nhất (chỉ cập nhật khi **lớn hơn hẳn** để lấy `k` nhỏ nhất).

# 17. Đáp án tham khảo

**Bài 1:** `nums = [3, 1, 2, 5]`, `total = 11`, `half = 2`.

| k | Mảng | Nửa đầu | Nửa sau | Tốt? |
|---|---|---:|---:|---|
| 0 | `[3, 1, 2, 5]` | 4 | 7 | ✘ |
| 1 | `[1, 2, 5, 3]` | 3 | 8 | ✘ |
| 2 | `[2, 5, 3, 1]` | 7 | 4 | ✔ |
| 3 | `[5, 3, 1, 2]` | 8 | 3 | ✔ |

Đáp án `2`. Mẹo cặp: `k = 0` (4 ≠ 7) và `k = 1` (3 ≠ 8) đều khác → `2` ✔.

**Bài 2:** `[4, 1, 2, 3]`, `[2, 4, 9]`, `[3, 4, 5, 0, 1, 2]`.

In [21]:
assert sol.countGoodRotations([3, 1, 2, 5]) == 2
print("Đáp án bài 1 đã được kiểm chứng ✔")

Đáp án bài 1 đã được kiểm chứng ✔


In [22]:
def answer_3_sliding(nums):
    n, half, total = len(nums), len(nums) // 2, sum(nums)
    first = sum(nums[:half])
    count = 0
    for k in range(n):
        if 2 * first > total:
            count += 1
        first += nums[(k + half) % n] - nums[k]
    return count

assert answer_3_sliding([1, 2, 3, 4, 5, 6]) == 3
assert answer_3_sliding([1, 2, 1, 2]) == 0
print("Đáp án bài 3 PASS")

Đáp án bài 3 PASS


In [23]:
def answer_4_pairs(nums):
    half, total = len(nums) // 2, sum(nums)
    first = sum(nums[:half])
    count = 0
    for k in range(half):
        if 2 * first != total:
            count += 1
        first += nums[k + half] - nums[k]
    return count

random.seed(4)
for _ in range(1000):
    nums = [random.randint(1, 5) for _ in range(2 * random.randint(1, 6))]
    assert answer_4_pairs(nums) == count_good_bruteforce(nums)
print("Đáp án bài 4 PASS")

Đáp án bài 4 PASS


In [24]:
def answer_5_at_least(nums):
    half, total = len(nums) // 2, sum(nums)
    first = sum(nums[:half])
    equal_pairs = 0
    for k in range(half):
        if 2 * first == total:
            equal_pairs += 1
        first += nums[k + half] - nums[k]
    return half + equal_pairs

def at_least_bruteforce(nums):
    n, half, total = len(nums), len(nums) // 2, sum(nums)
    return sum(1 for k in range(n) if 2 * sum((nums[k:] + nums[:k])[:half]) >= total)

random.seed(5)
for _ in range(1000):
    nums = [random.randint(1, 5) for _ in range(2 * random.randint(1, 6))]
    assert answer_5_at_least(nums) == at_least_bruteforce(nums)
print("Đáp án bài 5 PASS:", answer_5_at_least([1, 2, 1, 2]))

Đáp án bài 5 PASS: 4


In [25]:
def answer_6_best_rotation(nums):
    n, half = len(nums), len(nums) // 2
    first = sum(nums[:half])
    best_first, best_k = first, 0
    for k in range(n):
        if first > best_first:
            best_first, best_k = first, k
        first += nums[(k + half) % n] - nums[k]
    return best_k

def best_rotation_bruteforce(nums):
    n, half = len(nums), len(nums) // 2
    diffs = []
    for k in range(n):
        r = nums[k:] + nums[:k]
        diffs.append(sum(r[:half]) - sum(r[half:]))
    return diffs.index(max(diffs))           # index trả về vị trí ĐẦU TIÊN

assert answer_6_best_rotation([1, 2, 3, 4, 5, 6]) == 3
random.seed(6)
for _ in range(1000):
    nums = [random.randint(1, 5) for _ in range(2 * random.randint(1, 6))]
    assert answer_6_best_rotation(nums) == best_rotation_bruteforce(nums)
print("Đáp án bài 6 PASS")

Đáp án bài 6 PASS


# 18. Tổng kết

## Kiến thức Python

```python
nums[k:] + nums[:k]          # xoay vòng
i % n                        # chỉ số vòng tròn
nums + nums                  # nhân đôi mảng
from itertools import accumulate   # tổng cộng dồn
2 * first > total            # tránh chia số thực
```

## Kỹ thuật thuật toán

1. **Đổi hai tổng thành một** nhờ `total` không đổi.
2. **Sliding window vòng tròn**: bỏ `nums[k]`, thêm `nums[(k + half) % n]`.
3. **Prefix sum trên mảng nhân đôi**: `first(k) = P[k + half] - P[k]`.
4. **Đối xứng**: xoay `k` và `k + n/2` hoán đổi hai nửa → đáp án = số `k < n/2` có hai nửa khác nhau.

## Công thức nhớ nhanh

> Kiểm tra `2 * first > total`, rồi `first += nums[(k + n/2) % n] - nums[k]`.

## Độ phức tạp

```text
Time:  O(n)
Space: O(1)
```

## Checklist trước khi nộp

- [ ] So sánh **nghiêm ngặt** `>`?
- [ ] Dùng `% n` khi trượt (hoặc chỉ duyệt nửa đầu với mẹo cặp)?
- [ ] Kiểm tra **trước**, trượt **sau**?
- [ ] Không chia số thực?

# 19. Phiên bản cuối cùng nên nhớ

```python
class Solution:
    def countGoodRotations(self, nums: list[int]) -> int:
        n = len(nums)
        half = n // 2
        total = sum(nums)
        first = sum(nums[:half])
        count = 0
        for k in range(n):
            if 2 * first > total:
                count += 1
            first += nums[(k + half) % n] - nums[k]
        return count
```

Và phiên bản "thông minh" chỉ cần nửa vòng lặp:

```python
def countGoodRotations(nums):
    half, total = len(nums) // 2, sum(nums)
    first, count = sum(nums[:half]), 0
    for k in range(half):
        count += (2 * first != total)
        first += nums[k + half] - nums[k]
    return count
```